In [1]:
%config Completer.use_jedi = False

In [2]:
# resact function
import mariadb
import sys
import pandas as pd
import datetime
import itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.family'] = 'NanumGothic'


In [3]:
# resact function
import mariadb
import sys
import pandas as pd
import datetime
import itertools
import numpy as np
import matplotlib.pyplot as plt

def get_csv(table_name, gugun='춘천시'):
    try:
        conn = mariadb.connect(
        user = "root",
        password = "133nplab",
        host = "localhost",
        port = 3306,
        database = "TC"
        )
    except mariadb.Error as e:
        print(e)
        
    cur = conn.cursor()
    sql_query = "SELECT * from " + table_name
    cur.execute(sql_query)
    rows = cur.fetchall()
    table_query = "DESC " + table_name
    cur.execute(table_query)
    schema = cur.fetchall()
    columns = [name[0] for name in schema]
    recs = [rec for rec in rows]
    df = pd.DataFrame(recs,columns=columns)
    del rows
    return df[df['OCC_GUGUN_NM']== gugun]
    
def Format(time):
    try:
        tmp = datetime.datetime.strptime(str(time),'%Y%m%d%H%M%f')
        d = datetime.date(tmp.year , tmp.month, tmp.day)
        t = datetime.time(tmp.hour, tmp.minute)
        return datetime.datetime.combine(d, t)
    except:
        tmp = datetime.datetime.strptime(str(time),'%Y-%m-%d %H%M%f')
        d = datetime.date(tmp.year , tmp.month, tmp.day)
        t = datetime.time(tmp.hour, tmp.minute)
        return datetime.datetime.combine(d, t)

def hourFormat(time):
    try:
        tmp = datetime.datetime.strptime(str(time),'%Y-%m-%d %H:%M:%f').replace(minute=0)
        return tmp
    except:
        tmp = datetime.datetime.strptime(str(time),'%Y%m%d%H%M%f').replace(minute=0)
        return tmp
    
    
adam = ['신북읍','동면','동산면','신동면','동내면','남면','남산면',
         '서면','사북면','북산면','소양동','교동','조운동','약사명동','근화동',
         '후평1동','후평2동','후평3동','효자1동','효자2동','효자3동','석사동','퇴계동','강남동','신사우동']

etc_h_dong = {'소양동' : ['소양로2가' , '소양로3가' , '소양로4가' , '봉의동' , '옥천동' , '요선동' , '낙원동' , '중앙로1가'],
              '신사우동' : ['사농동' , '신동' , '우두동'],
              '근화동' : ['소양로1가' ,'중도동'],
              '약사명동' : ['약사동' , '중앙로2가' , '중앙로3가' , '죽림동'],
              '조운동' : ['조양동' , '운교동'],
              '후평동' : ['후평1동' , '후평2동' , '후평3동'],
              '강남동' : ['삼천동' , '송암동' , '온의동' , '칠전동'],
              '효자1동' : ['효자동'],
              '효자2동' : ['효자동'],
              '효자3동' : ['효자동'],
              '후평1동' : ['후평동'],
              '후평2동' : ['후평동'],
              '후평3동' : ['후평동'],
             }
h_dong = ['신북읍', '동면', '동산면', '신동면', '동내면', '남면', '남산면', '서면', '사북면', '북산면', '소양동', '교동', '조운동', '약사명동', 
          '근화동', '후평1동', '후평2동', '후평3동', '효자1동', '효자2동', '효자3동', '석사동', '퇴계동', '강남동', '신사우동']

def get_h_dong(dong_nm):
    if dong_nm in h_dong:
        return dong_nm
    else:
        for key in etc_h_dong.keys():
            if dong_nm in etc_h_dong[key]: 
                return key
            
def nm_space(dong_nm):
    if len(dong_nm) == 2:
        return f'{dong_nm[0]}  {dong_nm[1]}'
    else:
        return dong_nm

# df에서 원하는 기간 선택
def df_count(start , end , df , time_columns):
    '''
    start = Format(20211201000000)
    end   = Format(20211215000000)
    '''
    start = Format(start)
    end   = Format(end)

    time_line = pd.date_range(start=start, end=end , freq='1h')  
    in_time = df[df['REG_DTIME'].isin(time_line)]
    return in_time , time_line


In [4]:
class processing_data():
    def __init__(self, gugun = '춘천시'):
        self.data = get_csv("tc_resact", gugun)
        self.data['h_dong'] = self.data['OCC_DONG_NM'].apply(lambda x : get_h_dong(x))
        self.data['h_dong'] = self.data['h_dong'].apply(lambda x : nm_space(x))
        for idx in self.data.index:
            self.data.loc[idx ,'REG_DTIME'] = self.data['DSP_DAY'].loc[idx] + self.data['DSP_TIME'].loc[idx]
        self.data['REG_DTIME'] = self.data['REG_DTIME'].apply(lambda x : hourFormat(x))
        self.data = self.data[['REG_DTIME' , 'ACT_DESC' , 'GUIDE', 'h_dong']]
        self.data.sort_values(by = 'REG_DTIME', inplace = True)
        
        
    def select_time(self, start = 20100101000000, end = 20220401000000):
        in_time , time_line = df_count(start, end , self.data , 'REG_DTIME' )
        
        in_time.sort_values(by = 'REG_DTIME', inplace = True)
        return in_time , time_line
    
    def get_count(self):
        in_time, time_line = self.select_time()
        time_line = pd.date_range('2010/01/01' , '2022/03/01', freq='1h')#.astype(str)
        h_dong_idx = in_time['h_dong'].unique()
        mulit_index = list(itertools.product(time_line , h_dong_idx))
        #print(mulit_index)
        count_df = pd.DataFrame(columns = ['REG_DTIME' , 'h_dong' , 'count' , 'ACT_DESC' , 'GUIDE'])
        IDX = 0
        SUM = 0
        for time , h_dong in mulit_index:
        
            df_idxs = (in_time['REG_DTIME'] == time) & (in_time['h_dong'] == h_dong)
            if sum(df_idxs) == 0:
                count_df.loc[IDX] = [time , h_dong ,sum(df_idxs) , None , None]
            else:
                count_df.loc[IDX] = [time , h_dong ,sum(df_idxs) , list(in_time[df_idxs]['ACT_DESC']) , list(in_time[df_idxs]['GUIDE'])]
                SUM += sum(df_idxs)
                print(f'\n ---------- {time} : {h_dong}   {SUM} ---------- \n')
            IDX += 1
        
        return count_df

            
    def get_resact(self, path = 'resact_count.csv'):
        df =  pd.read_csv(path)
        count_df = df[['REG_DTIME', 'h_dong' , 'count']]        
        self.count_df = count_df 
        return count_df
    
    def orgnal_count_plot(self , year):
        df = self.get_resact()
        dongs = df['h_dong'].unique()
        year_s = f'{year}/1/1/00'
        year_e = f'{year+1}/1/1/00'
        dt = pd.date_range(year_s ,year_e, freq='h')[:-1]        
        year_df = df[df['REG_DTIME'].isin(dt.astype(str))]
        f , axes = plt.subplots(21,1 , figsize=(20,66))
        x = range(len(year_df[year_df['h_dong'] == '강남동']['count']))
        for i,dong_nm in enumerate(df['h_dong'].unique()):
            year_dong_df = year_df[year_df['h_dong'] == dong_nm]['count']
            rat = len(year_dong_df[year_dong_df > 0]) / len(year_dong_df)
            axes[i].set_title(f'{dong_nm} count in {year} , ratio : {rat:.4f}')
            axes[i].scatter(x,year_dong_df , s=1)
        plt.show()



In [5]:
pdata  = processing_data()
data = pdata.get_count()

#pdata.orgnal_count_plot(2021)

<ipython-input-4-5634852b3d66>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  in_time.sort_values(by = 'REG_DTIME', inplace = True)



 ---------- 2010-01-01 00:00:00 : 석사동   1 ---------- 


 ---------- 2010-01-01 06:00:00 : 신동면   2 ---------- 


 ---------- 2010-01-01 09:00:00 : 강남동   3 ---------- 


 ---------- 2010-01-01 10:00:00 : 강남동   4 ---------- 


 ---------- 2010-01-01 10:00:00 : 서  면   5 ---------- 


 ---------- 2010-01-01 14:00:00 : 남  면   7 ---------- 


 ---------- 2010-01-02 13:00:00 : 서  면   9 ---------- 


 ---------- 2010-01-02 17:00:00 : 사북면   10 ---------- 


 ---------- 2010-01-02 19:00:00 : 동내면   11 ---------- 


 ---------- 2010-01-02 22:00:00 : 효자1동   12 ---------- 


 ---------- 2010-01-02 22:00:00 : 남산면   14 ---------- 


 ---------- 2010-01-03 06:00:00 : 남산면   15 ---------- 


 ---------- 2010-01-03 19:00:00 : 근화동   16 ---------- 


 ---------- 2010-01-03 23:00:00 : 효자1동   17 ---------- 


 ---------- 2010-01-04 10:00:00 : 신사우동   18 ---------- 


 ---------- 2010-01-04 11:00:00 : 남  면   21 ---------- 


 ---------- 2010-01-04 15:00:00 : 근화동   22 ---------- 


 ---------- 2010-01-04 16:00:0


 ---------- 2010-02-17 10:00:00 : 근화동   169 ---------- 


 ---------- 2010-02-17 16:00:00 : 서  면   171 ---------- 


 ---------- 2010-02-17 20:00:00 : 약사명동   172 ---------- 


 ---------- 2010-02-18 00:00:00 : 남산면   173 ---------- 


 ---------- 2010-02-18 19:00:00 : 남  면   175 ---------- 


 ---------- 2010-02-19 02:00:00 : 석사동   176 ---------- 


 ---------- 2010-02-19 19:00:00 : 근화동   177 ---------- 


 ---------- 2010-02-19 21:00:00 : 근화동   178 ---------- 


 ---------- 2010-02-20 01:00:00 : 석사동   179 ---------- 


 ---------- 2010-02-20 08:00:00 : 신사우동   180 ---------- 


 ---------- 2010-02-20 09:00:00 : 신사우동   181 ---------- 


 ---------- 2010-02-20 11:00:00 : 강남동   182 ---------- 


 ---------- 2010-02-21 04:00:00 : 후평1동   183 ---------- 


 ---------- 2010-02-21 22:00:00 : 석사동   184 ---------- 


 ---------- 2010-02-22 10:00:00 : 북산면   185 ---------- 


 ---------- 2010-02-22 12:00:00 : 후평1동   186 ---------- 


 ---------- 2010-02-22 16:00:00 : 남산면   187 ---------- 


 -----


 ---------- 2010-04-06 13:00:00 : 남산면   323 ---------- 


 ---------- 2010-04-06 13:00:00 : 퇴계동   324 ---------- 


 ---------- 2010-04-06 14:00:00 : 후평1동   325 ---------- 


 ---------- 2010-04-07 00:00:00 : 사북면   326 ---------- 


 ---------- 2010-04-07 14:00:00 : 남  면   328 ---------- 


 ---------- 2010-04-08 14:00:00 : 퇴계동   329 ---------- 


 ---------- 2010-04-08 17:00:00 : 후평1동   330 ---------- 


 ---------- 2010-04-08 23:00:00 : 석사동   331 ---------- 


 ---------- 2010-04-09 17:00:00 : 신사우동   332 ---------- 


 ---------- 2010-04-09 21:00:00 : 효자1동   333 ---------- 


 ---------- 2010-04-10 04:00:00 : 근화동   334 ---------- 


 ---------- 2010-04-10 05:00:00 : 근화동   335 ---------- 


 ---------- 2010-04-10 08:00:00 : 석사동   336 ---------- 


 ---------- 2010-04-11 14:00:00 : 신북읍   337 ---------- 


 ---------- 2010-04-11 20:00:00 : 근화동   338 ---------- 


 ---------- 2010-04-12 18:00:00 : 퇴계동   339 ---------- 


 ---------- 2010-04-13 22:00:00 : 석사동   340 ---------- 


 -------


 ---------- 2010-05-12 17:00:00 : 남산면   478 ---------- 


 ---------- 2010-05-12 18:00:00 : 남산면   479 ---------- 


 ---------- 2010-05-12 23:00:00 : 효자1동   480 ---------- 


 ---------- 2010-05-13 13:00:00 : 신사우동   481 ---------- 


 ---------- 2010-05-13 15:00:00 : 후평1동   482 ---------- 


 ---------- 2010-05-14 14:00:00 : 신사우동   483 ---------- 


 ---------- 2010-05-14 20:00:00 : 남  면   484 ---------- 


 ---------- 2010-05-15 02:00:00 : 퇴계동   485 ---------- 


 ---------- 2010-05-15 09:00:00 : 동내면   486 ---------- 


 ---------- 2010-05-15 11:00:00 : 남산면   487 ---------- 


 ---------- 2010-05-16 02:00:00 : 동내면   488 ---------- 


 ---------- 2010-05-16 09:00:00 : 동내면   489 ---------- 


 ---------- 2010-05-16 14:00:00 : 북산면   492 ---------- 


 ---------- 2010-05-16 18:00:00 : 후평1동   493 ---------- 


 ---------- 2010-05-17 13:00:00 : 후평1동   494 ---------- 


 ---------- 2010-05-17 17:00:00 : 교  동   495 ---------- 


 ---------- 2010-05-17 22:00:00 : 남  면   496 ---------- 


 ---

...


 ---------- 2021-12-20 18:00:00 : 퇴계동   39698 ---------- 


 ---------- 2021-12-20 19:00:00 : 사북면   39699 ---------- 


 ---------- 2021-12-20 19:00:00 : 동  면   39700 ---------- 


 ---------- 2021-12-20 22:00:00 : 신동면   39701 ---------- 


 ---------- 2021-12-21 11:00:00 : 동내면   39702 ---------- 


 ---------- 2021-12-21 11:00:00 : 신사우동   39704 ---------- 


 ---------- 2021-12-21 13:00:00 : 동산면   39705 ---------- 


 ---------- 2021-12-21 17:00:00 : 교  동   39706 ---------- 


 ---------- 2021-12-21 19:00:00 : 석사동   39707 ---------- 


 ---------- 2021-12-21 20:00:00 : 강남동   39708 ---------- 


 ---------- 2021-12-22 09:00:00 : 강남동   39709 ---------- 


 ---------- 2021-12-22 10:00:00 : 강남동   39711 ---------- 


 ---------- 2021-12-22 10:00:00 : 동산면   39712 ---------- 


 ---------- 2021-12-22 11:00:00 : 후평1동   39713 ---------- 


 ---------- 2021-12-22 13:00:00 : 소양동   39714 ---------- 


 ---------- 2021-12-22 14:00:00 : 동  면   39715 ---------- 


 ---------- 2021-12-22 16:00:00 : 


 ---------- 2022-01-01 23:00:00 : 근화동   39886 ---------- 


 ---------- 2022-01-02 01:00:00 : 퇴계동   39887 ---------- 


 ---------- 2022-01-02 03:00:00 : 퇴계동   39888 ---------- 


 ---------- 2022-01-02 10:00:00 : 효자1동   39889 ---------- 


 ---------- 2022-01-02 13:00:00 : 동  면   39890 ---------- 


 ---------- 2022-01-02 17:00:00 : 효자1동   39891 ---------- 


 ---------- 2022-01-02 18:00:00 : 석사동   39892 ---------- 


 ---------- 2022-01-02 18:00:00 : 후평1동   39893 ---------- 


 ---------- 2022-01-03 00:00:00 : 신사우동   39895 ---------- 


 ---------- 2022-01-03 05:00:00 : 동내면   39896 ---------- 


 ---------- 2022-01-03 06:00:00 : 석사동   39897 ---------- 


 ---------- 2022-01-03 07:00:00 : 동내면   39898 ---------- 


 ---------- 2022-01-03 09:00:00 : 약사명동   39899 ---------- 


 ---------- 2022-01-03 10:00:00 : 동  면   39900 ---------- 


 ---------- 2022-01-03 11:00:00 : 근화동   39901 ---------- 


 ---------- 2022-01-03 11:00:00 : 동  면   39902 ---------- 


 ---------- 2022-01-03 11:00:00


 ---------- 2022-01-15 09:00:00 : 신북읍   40048 ---------- 


 ---------- 2022-01-15 11:00:00 : 신동면   40049 ---------- 


 ---------- 2022-01-15 14:00:00 : 후평1동   40050 ---------- 


 ---------- 2022-01-15 18:00:00 : 신북읍   40051 ---------- 


 ---------- 2022-01-15 23:00:00 : 효자1동   40053 ---------- 


 ---------- 2022-01-16 11:00:00 : 동내면   40054 ---------- 


 ---------- 2022-01-16 13:00:00 : 퇴계동   40055 ---------- 


 ---------- 2022-01-16 14:00:00 : 남산면   40057 ---------- 


 ---------- 2022-01-16 15:00:00 : 퇴계동   40058 ---------- 


 ---------- 2022-01-16 17:00:00 : 퇴계동   40059 ---------- 


 ---------- 2022-01-16 20:00:00 : 후평1동   40060 ---------- 


 ---------- 2022-01-17 00:00:00 : 석사동   40061 ---------- 


 ---------- 2022-01-17 03:00:00 : 사북면   40063 ---------- 


 ---------- 2022-01-17 13:00:00 : 강남동   40064 ---------- 


 ---------- 2022-01-17 13:00:00 : 동  면   40065 ---------- 


 ---------- 2022-01-17 15:00:00 : 강남동   40066 ---------- 


 ---------- 2022-01-17 15:00:00 : 조


 ---------- 2022-01-30 10:00:00 : 강남동   40238 ---------- 


 ---------- 2022-01-30 10:00:00 : 약사명동   40239 ---------- 


 ---------- 2022-01-30 19:00:00 : 강남동   40240 ---------- 


 ---------- 2022-01-30 20:00:00 : 사북면   40241 ---------- 


 ---------- 2022-01-30 20:00:00 : 효자1동   40242 ---------- 


 ---------- 2022-01-30 21:00:00 : 동내면   40244 ---------- 


 ---------- 2022-01-30 21:00:00 : 동  면   40245 ---------- 


 ---------- 2022-01-31 09:00:00 : 강남동   40247 ---------- 


 ---------- 2022-01-31 09:00:00 : 서  면   40249 ---------- 


 ---------- 2022-01-31 10:00:00 : 동산면   40250 ---------- 


 ---------- 2022-01-31 11:00:00 : 후평1동   40251 ---------- 


 ---------- 2022-01-31 12:00:00 : 석사동   40254 ---------- 


 ---------- 2022-01-31 12:00:00 : 남  면   40259 ---------- 


 ---------- 2022-01-31 13:00:00 : 강남동   40260 ---------- 


 ---------- 2022-01-31 14:00:00 : 남  면   40264 ---------- 


 ---------- 2022-01-31 18:00:00 : 퇴계동   40265 ---------- 


 ---------- 2022-01-31 19:00:00 


 ---------- 2022-02-20 21:00:00 : 퇴계동   40431 ---------- 


 ---------- 2022-02-20 22:00:00 : 석사동   40432 ---------- 


 ---------- 2022-02-20 23:00:00 : 사북면   40433 ---------- 


 ---------- 2022-02-21 08:00:00 : 신동면   40434 ---------- 


 ---------- 2022-02-21 14:00:00 : 퇴계동   40435 ---------- 


 ---------- 2022-02-21 15:00:00 : 동산면   40437 ---------- 


 ---------- 2022-02-21 16:00:00 : 남  면   40440 ---------- 


 ---------- 2022-02-21 18:00:00 : 퇴계동   40441 ---------- 


 ---------- 2022-02-21 20:00:00 : 동내면   40442 ---------- 


 ---------- 2022-02-22 18:00:00 : 소양동   40443 ---------- 


 ---------- 2022-02-22 18:00:00 : 후평1동   40444 ---------- 


 ---------- 2022-02-22 22:00:00 : 후평1동   40445 ---------- 


 ---------- 2022-02-23 07:00:00 : 서  면   40446 ---------- 


 ---------- 2022-02-23 10:00:00 : 신사우동   40447 ---------- 


 ---------- 2022-02-23 11:00:00 : 신사우동   40449 ---------- 


 ---------- 2022-02-23 14:00:00 : 강남동   40450 ---------- 


 ---------- 2022-02-23 16:00:00 :

In [14]:
data['REG_DTIME'].unique()

array(['2015-01-01T00:00:00.000000000', '2015-01-01T01:00:00.000000000',
       '2015-01-01T02:00:00.000000000', ...,
       '2022-02-28T22:00:00.000000000', '2022-02-28T23:00:00.000000000',
       '2022-03-01T00:00:00.000000000'], dtype='datetime64[ns]')

In [6]:
data.to_csv('2010_2022_count.csv')

In [11]:
data[data['count'] > 0]

,REG_DTIME,h_dong,count,ACT_DESC,GUIDE
189,2015-01-01 09:00:00,효자1동,1,[모닝차량이 나무와 충돌한 상황으로 현장도착한바 구조사항 없다하여 귀대함.],[None]
274,2015-01-01 13:00:00,석사동,1,[상기장소 시건개방 출동으로 자체처리 되었다는 신고로 출동전 취소됨. ],[None]
359,2015-01-01 17:00:00,후평1동,1,[상기장소 화재출동으로 현장 출동 하여 확인 중 춘천지휘에서 구조상황 없다하여 구조...,[None]
379,2015-01-01 18:00:00,석사동,1,[화재접보로 현장출동 중 자체진화되었다는 상황실무전을 받고 회차 귀서함.],[None]
715,2015-01-02 10:00:00,석사동,1,"[화재출동으로 현장으로 출동중 선착한 석사펌프에서 주택 내 쓰레기더미 화재이며, 구...",[None]
...,...,...,...,...,...
1317960,2022-02-28 00:00:00,효자1동,2,"[80대 노인 행방불명으로 출동지령나왔으나, 구조대 출동하지말라는 유선연락 수신 후...","[None, None]"
1318175,2022-02-28 10:00:00,신북읍,1,[- 고양이가 지붕위로 올라가서 못내려오고있다는 신고로 출동. - 현장도착하여 동...,[None]
1318208,2022-02-28 11:00:00,신사우동,1,[차량화재 신고로 출동중 아스팔트 녹이는 중이며 선착한 펌프에서 진압 가능하다하여 ...,[None]
1318298,2022-02-28 16:00:00,후평1동,2,[[위치추척] 아내 자살기도 추정 위치추적 출동. 출동 중 신북구급 출동으로 구조...,"[출동 중 귀소., None]"


In [21]:
pdata.data[pdata.data['REG_DTIME'] > '2022-01-01 00:00:00.000']

,REG_DTIME,ACT_DESC,GUIDE,h_dong
372165,2022-01-01 09:00:00,자동화재속보설비 출동건으로 현장 출동중 관계자에 의해 오작동 확인되어 출동중 귀소함,None,효자1동
372152,2022-01-01 09:00:00,소방시설오작동으로 춘천구조 안전하게 중도귀소완료.,소방시설 오작동으로 중도귀소.,효자1동
372162,2022-01-01 10:00:00,"쓰레기를 태우다 폐기물더미(냉장고, 침대 매트리스 등)으로 옮겨붙은 화재로 수관 2...",None,근화동
372161,2022-01-01 10:00:00,자동화재속보설비 작동으로 현장 도착하여 1층 처치실 차동식감지기 오작동 확인 화재징...,None,약사명동
372167,2022-01-01 10:00:00,어머니께서 말씀이 어눌하고 손 감각이 이상하다는 신고로 현장도착한바 스스로 걸어나오...,안전조치,석사동
...,...,...,...,...
377100,2022-02-28 10:00:00,- 고양이가 지붕위로 올라가서 못내려오고있다는 신고로 출동. - 현장도착하여 동물...,None,신북읍
377102,2022-02-28 11:00:00,차량화재 신고로 출동중 아스팔트 녹이는 중이며 선착한 펌프에서 진압 가능하다하여 지...,None,신사우동
377099,2022-02-28 16:00:00,[고양이 구조] 고양이가 본인 집 하수구에 끼어 못나온다는 신고. 구조대 현장 도...,배수구 뚜껑 개방,신동면
377103,2022-02-28 16:00:00,[위치추척] 아내 자살기도 추정 위치추적 출동. 출동 중 신북구급 출동으로 구조대...,출동 중 귀소.,후평1동


In [18]:
sum(data[data['count'] > 0]['count'])

613